# Create ETIS HTM Awards (Estonian Ministry of Education and Research)

Awards financed by the **Estonian Ministry of Education and Research**
(Haridus- ja Teadusministeerium, HTM; OpenAlex `F4320321976`, EE) from Estonia's
national research information system **ETIS**, via its key-less "ETIS2 Open
Data" REST API (`https://www.etis.ee:7443/api/project/getitems`, the same API
the ETAg/ETF ingests use). **2,143 in-scope projects, 1997-2026, EUR 492.9M;
100% title/dates/PI/institution, 94.2% amount, 91.7% abstract** (local run
2026-10-01; 26,597 ETIS projects pulled).

**Routing (runbook §2.3.2).** ETIS covers every Estonian funder, so the script
keeps only projects whose `FinancingInstitutions` include HTM (2,235), and routes
them: 2,221 to HTM; 7 co-financed by the Estonian Research Council / Estonian
Science Foundation are left to the existing `etis_etag` (priority 363) /
`etis_etf` (365) ingests; 6 co-financed by the European Commission / an EU
executive agency (Horizon Europe, CEF) are EU-led and skipped; 1 where another
financier has the larger published share is skipped.

**Scope filter (stated per the 2026-10-01 rule; `in_scope` column in the raw
table).** Dropped (78): ETIS project type *Õppearendusprojekt* (teaching/study
development: adult-education courses, preliminary vocational training in
gymnasiums, assistant-teacher training, youth competitions) and 3 *Muu*-type
rows under the generic ministry programme 200 (EdTech Hack, assistant
teachers). One teaching-development row that is a research survey (EU Kids
Online Survey in Estonia) is kept. KEPT and flagged: 396 research-infrastructure
grants (*Taristuprojekt*: equipment purchase support AP, TAP modernisation;
`funding_type = 'infrastructure'`), 59 government-commissioned studies
(`'contract'`), and the generic ministry programmes (codes 100/101/105/106/200,
MUU) whose rows ETIS files as R&D but which include some education-development
items, small events (hackathons, a school fair) and teacher-training projects.

**`funder_award_id` (§2.1.1).** ETIS `FinancierProjectNr` is the number citing
works write under this funder (e.g. targeted-financing themes `SF0180089s08`,
Centres of Excellence `TK117`/`TK218U1`, national programmes `EKKM14-300`), so it
is used for 1,588 awards when it looks like an identifier (leading "Nr" /
"Lepingu nr" stripped; must contain a digit). 555 awards get the synthetic key
`HTM-ETIS-{ETIS project GUID}`: no number (most), a free-text contract/decision
registration or bare word ("Töövõtuleping"), or a number shared by several ETIS records (largest-amount record
keeps the number; 10 rows). The raw number is kept in `financier_project_nr`.
285 of the 1,699 existing citation stubs on F4320321976 (123 distinct ids) carry an
id this ingest ships, so they collapse onto these awards. Most other stubs are
IUT/PUT/PRG/ETF numbers, which are Estonian Research Council / Science
Foundation grants (already ingested under F4320321090 / F4320321018).

**Amount.** `FinancingInPeriodsTotal` (EUR; ETIS converts pre-2011 EEK). For
co-financed projects the HTM `TotalFinancing` (12) or `Proportion` share is used;
15 co-financed rows with no HTM share published have `amount` NULL.
**Roof/sub-projects:** the 2024 Centres of Excellence (TK200-TK232) and two HARTA
consortia have a roof record (e.g. `TK200`, EUR 7.0M) plus one record per partner
(`TK200U1`..`U8`) whose amounts sum to the roof's. All records ship (partners
carry their own PI/institution; both id forms are cited), but the 88 partner
records under a shipped roof have `amount` NULL (own share kept in the raw
`project_total_eur`) so the ministry's total is not double counted.

**PI.** First ETIS "Responsible executor" (split with the canonical
`split_name`); a second one is `co_lead_investigator`; all are in
`investigators`. Affiliation = coordinator institution (country `EE` when it has
an 8-digit Estonian business-register code). ETIS publishes no ORCIDs here.

**Access note.** www.etis.ee and the :7443 API host have a generic
`User-agent: * / Disallow: /` robots.txt (no AI-specific rule; www.etis.ee
allows `/Portal/`). The API is ETIS's published open-data interface and is the
source of the existing ETAg/ETF ingests. Landing pages are
`https://www.etis.ee/Portal/Projects/Display/{guid}`.

**Prerequisites:** run `scripts/local/etis_htm_to_s3.py` (uploads
`s3://openalex-ingest/awards/etis_htm/etis_htm_projects.parquet`, §1.4 shrink
guard).

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320321976`,
Haridus- ja Teadusministeerium, ROR 051aqjh92, DOI 10.13039/501100003510 (read
from `openalex.funders.funders` by ID).

**Priority:** `590` (direct funder ingest; higher wins under the 2026-06-20 DESC
dedup). Provenance `etis_htm`.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.etis_htm_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/etis_htm/etis_htm_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN LOWER(in_scope) = 'true' THEN 1 ELSE 0 END) as in_scope_rows,
       COUNT(DISTINCT etis_guid) as distinct_guids
FROM openalex.awards.etis_htm_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.etis_htm_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.etis_htm_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321976 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320321976;

## Step 2: Create ETIS HTM Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.etis_htm_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320321976  -- Haridus- ja Teadusministeerium
),

g AS (
    SELECT *,
        from_json(responsible_executors_json,
                  'ARRAY<STRUCT<name:STRING, given_name:STRING, family_name:STRING>>') AS rx
    FROM openalex.awards.etis_htm_raw
    WHERE LOWER(in_scope) = 'true'       -- scope filter, see header
      AND route = 'htm'
      AND funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
),

people AS (
    -- Responsible executors in ETIS order. Struct field order MUST match
    -- openalex_awards_raw: given_name, family_name, orcid, role_start, affiliation.
    -- Affiliation (coordinator institution) only on the first person.
    SELECT
        etis_guid,
        TRANSFORM(COALESCE(rx, ARRAY()), (p, i) -> named_struct(
            'given_name', NULLIF(TRIM(p.given_name), ''),
            'family_name', NULLIF(TRIM(p.family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', CASE WHEN i = 0 THEN NULLIF(TRIM(institution), '') END,
                'country', CASE WHEN i = 0 THEN NULLIF(TRIM(institution_country), '') END,
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )) AS investigators
    FROM g
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) END as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    NULLIF(TRIM(g.programme_name), '') as funder_scheme,
    'etis_htm' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
JOIN people p ON p.etis_guid = g.etis_guid
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'etis_htm' AND priority = 590;

-- Insert into openalex_awards_raw with priority.
-- Priority 590: direct-from-funder ingest (ETIS, Ministry of Education and
-- Research projects). Higher wins under the 2026-06-20 DESC dedup (oxjob #500),
-- so it outranks the acknowledgement shells (priority 0) on the same award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    590 as priority
FROM openalex.awards.etis_htm_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       COUNT(DISTINCT LOWER(funder_award_id)) as distinct_award_ids
FROM openalex.awards.etis_htm_awards;

In [ ]:
%sql
-- §2.1.1 shape check: ids are either an ETIS FinancierProjectNr (no whitespace)
-- or the synthetic HTM-ETIS-{guid}; expect 0 rows with whitespace (no 360Giving-style registry ids).
SELECT
    SUM(CASE WHEN funder_award_id LIKE 'HTM-ETIS-%' THEN 1 ELSE 0 END) as synthetic_ids,
    SUM(CASE WHEN funder_award_id NOT LIKE 'HTM-ETIS-%' THEN 1 ELSE 0 END) as financier_project_nr_ids,
    SUM(CASE WHEN instr(funder_award_id, ' ') > 0 OR instr(funder_award_id, ';') > 0 THEN 1 ELSE 0 END) as ids_with_space_or_list
FROM openalex.awards.etis_htm_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency,
       start_date, end_date, lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, co_lead_investigator.family_name, landing_page_url
FROM openalex.awards.etis_htm_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.etis_htm_awards
GROUP BY funder_scheme, funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.etis_htm_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.etis_htm_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Title frequency: repeated generic titles are expected for equipment grants
-- ("Modernization of small-scale research infrastructure") and Centre of
-- Excellence partner records (one ETIS record per partner, TKnnnUk numbers).
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.etis_htm_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect ~100% title/dates/PI, ~94% amount, ~92% abstract).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.etis_htm_awards;

In [ ]:
%sql
-- Citation stubs on F4320321976 that this ingest upgrades (same award id).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as upgraded_by_etis_htm
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.etis_htm_awards t ON t.id = c.id
WHERE c.funder_id = 4320321976 AND c.priority = 0
GROUP BY c.provenance ORDER BY stub_rows DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'etis_htm'
GROUP BY provenance, priority;